In [ ]:
import joblib
import pandas as pd
import numpy as np
import random
from faker import Faker

# 1. 모델 불러오기
lgbm_model = joblib.load('C:\skn24\2차프로젝트\SKN24-2nd-4Team\data\model job\lgbm_model.joblib.pkl')

# 2. 카테고리 매핑 정보
CATEGORY_MAP = {
    '멤버십상태': {'ACTIVE': 0, 'LEFT CLUB': 1, 'PRE-CREATE': 2},
    '연령대': {'10대': 0, '20대': 1, '30대': 2, '40대': 3, '50대': 4, '60대': 5, '70대 이상': 6, 'UNKNOWN': 7},
    '패션뉴스구독여부': {'N': 0, 'Y': 1},
    '상품그룹': {
        'Accessories': 0, 'Bags': 1, 'Cosmetic': 2, 'Fun': 3, 'Furniture': 4,
        'Garment Full body': 5, 'Garment Lower body': 6, 'Garment Upper body': 7,
        'Garment and Shoe care': 8, 'Interior textile': 9, 'Items': 10, 'Nightwear': 11,
        'Shoes': 12, 'Socks & Tights': 13, 'Stationery': 14, 'Swimwear': 15,
        'Underwear': 16, 'Underwear/nightwear': 17, 'Unknown': 18
    }
}

# 3. [직접 입력] 사용자님이 주신 5명의 데이터
input_data = [
    ["정재훈", "Bags", "ACTIVE", "30대", 0.054, "Y"],
    ["조아름", "Cosmetic", "LEFT CLUB", "20대", 0.120, "N"],
    ["정석원", "Shoes", "ACTIVE", "20대", 0.003, "Y"],
    ["김민준", "Accessories", "PRE-CREATE", "20대", 0.850, "N"],
    ["정준하", "Underwear", "ACTIVE", "20대", 0.021, "Y"]
]

# 4. [Faker 추가] 추가로 10명의 가짜 데이터를 생성하여 합치기
fake = Faker('ko-KR')
for _ in range(10): # 10명을 더 만들고 싶다면 이 숫자를 조절하세요
    input_data.append([
        fake.name(), 
        random.choice(list(CATEGORY_MAP['상품그룹'].keys())),
        random.choice(list(CATEGORY_MAP['멤버십상태'].keys())),
        random.choice(list(CATEGORY_MAP['연령대'].keys())),
        round(np.random.beta(0.5, 2), 4), # 정규화된 가격 (낮은 값 위주)
        random.choice(['Y', 'N'])
    ])

# 5. 예측 실행 함수
def get_final_predictions(data_list, lgbm, xgb, gb):
    columns = ['이름', '상품그룹', '멤버십상태', '연령대', '가격', '패션뉴스구독여부']
    df = pd.DataFrame(data_list, columns=columns)
    
    # 전처리 (문자열 -> 숫자 매핑)
    X_input = df.copy()
    for col, mapping in CATEGORY_MAP.items():
        if col in X_input.columns:
            X_input[col] = X_input[col].map(mapping)
            X_input[col] = X_input[col].astype('category') # LightGBM 필수 설정
            
    # 피처 추출 (이름 제외)
    X_features = X_input.drop('이름', axis=1)
    
    # 모델별 최종 분류 결과 (0: 유지, 1: 이탈)
    res = df.copy()
    res['LGBM_판단'] = ['이탈' if x == 1 else '유지' for x in lgbm.predict(X_features)]
    
    return res

# 실행 및 결과 출력
final_result = get_final_predictions(input_data, lgbm_model)

print("\n--- 🔍 [실제 5명 + Faker 10명] 통합 예측 결과 ---")
print(final_result)

FileNotFoundError: [Errno 2] No such file or directory: 'C:/skn24/2차프로젝트/SKN24-2nd-4Team/data/model job/lgbm_model.joblib.pkl'